# Project Milestone: "Watts the Cost?" — Data Pipeline & Feature Engineering
**Course:** UC Berkeley DATASCI 207 — Summer 2026  
**Team Members:** Julia Craciun, Roe Merry Jr., Ricardo Peralta, Jessie Zhou  
**Repository:** [watts-the-cost-DATASCI-207](https://github.com/jcraciun/watts-the-cost-DATASCI-207)

---

## 🏛️ 1. Project Motivation & Abstract
The Energy Information Agency (EIA) recently reported a retail electricity rate surge exceeding 5% year-over-year. Driven by rigid, non-discretionary household demands and accelerating data center grid infrastructure, electricity costs represent an escalating economic equity burden. 

Our pipeline investigates structural, macroeconomic, utility-governed, and demographic features at the U.S. Zip Code level to uncover not just *where* utility rates are fluctuating, but *why*. We explicitly parse out the interactions between community income distributions, localized commercial presence, and utility ownership structures (Investor-Owned Utilities vs. Public Cooperatives).

## 🔌 Phase 1: Ingestion, Parsing, & Cross-Dataset Verification
Before conducting modeling experiments, we isolate and verify our baseline pipeline architecture. This section implements and cross-validates the relational data assembly strategy initially structured by team member Roe Merry Jr. 

### Core Extraction Architecture:
1. **Demographic Ingestion:** Extracts the American Community Survey (ACS) 5-Year Income Estimates (`ACSST5Y2023.S1901-Data.csv`).
2. **Economic Fingerprinting:** Processes the Zip Code Business Patterns archive (`zbp23detail.zip`) to index local business densities.
3. **Utility Rate Melting:** Ingests separate infrastructure matrices (`iou_zipcodes_2023.csv` and `non_iou_zipcodes_2023.csv`) and passes them to a high-performance **DuckDB** engine to melt commercial, industrial, and residential pricing structures into a single continuous target vector (`rate`).
4. **Geographic Standardization:** Enforces standard string padding (`str.zfill(5)`) to align 5-digit U.S. Zip Code formats across all disparate tables.

In [ ]:
import duckdb
from pathlib import Path
import pandas as pd

# Coordinate localized directories
current_dir = Path.cwd()
data_dir = current_dir.joinpath("data")
print(f"Data Source Environment: {data_dir}")

# Ingest original uncompressed source archives
income_df_raw = pd.read_csv(data_dir.joinpath("ACSST5Y2023.S1901-Data.csv"), sep=",", encoding='latin-1')
business_df_raw = pd.read_csv(data_dir.joinpath("zbp23detail.zip"), sep=",", encoding='latin-1')
iou_df_raw = pd.read_csv(data_dir.joinpath("iou_zipcodes_2023.csv"), sep=",", encoding='latin-1')
non_iou_df_raw = pd.read_csv(data_dir.joinpath("non_iou_zipcodes_2023.csv"), sep=",", encoding='latin-1')

# Process and standardize the geographic Census Income matrix
income_df = income_df_raw.copy()
income_df.columns = income_df.iloc[0]
income_df = income_df[1:]
income_df.reset_index(drop=True, inplace=True)
income_df.columns.name = None

income_df_short = income_df.iloc[:, :29].copy()
income_df_short.insert(0, 'zip', income_df_short['Geographic Area Name'].astype(str).str[-5:])
income_df_short.drop(columns=['Geography', 'Geographic Area Name'], axis=1, inplace=True)

rename_substring_map = {
    'Estimate': '', 'Margin of Error': 'moe', 'Total': '', 
    '!!': '_', '(dollars)': '', ' ': '_', ',999': 'k', ',000': 'k'
}
rename_dict = {}
for column in income_df_short.columns.copy():
    new_column_name = column
    for old_substr, new_substr in rename_substring_map.items():
        if old_substr in new_column_name:
            new_column_name = new_column_name.replace(old_substr, new_substr)
    while '__' in new_column_name:
        new_column_name = new_column_name.replace('__', '_')
    new_column_name = new_column_name.strip('_')
    rename_dict[column] = new_column_name.lower()

income_df_short.rename(columns=rename_dict, inplace=True)
income_df_short = income_df_short.apply(pd.to_numeric, errors='coerce')
income_df_final = income_df_short.dropna().copy()
income_df_final['zip'] = income_df_final['zip'].astype(int).astype(str).str.zfill(5)

# Standardize and aggregate Census Business Patterns
business_df = business_df_raw.copy()
business_df.insert(1, 'sector', business_df['naics'].astype(str).str[:2])
business_df.drop(columns=['name', 'city', 'stabbr', 'cty_name', 'naics'], axis=1, inplace=True)
business_df = business_df.apply(pd.to_numeric, errors='coerce').fillna(0)
business_df_final = business_df.drop(columns=['sector'], axis=1).groupby(['zip'], as_index=False).sum()
business_df_final['zip'] = business_df_final['zip'].astype(int).astype(str).str.zfill(5)

# Mirror and prepare baseline utility structures
iou_df = iou_df_raw.copy()
non_iou_df = non_iou_df_raw.copy()
iou_df['zip'] = iou_df['zip'].astype(int).astype(str).str.zfill(5)
non_iou_df['zip'] = non_iou_df['zip'].astype(int).astype(str).str.zfill(5)

# Build multi-table relational schema strings
inc_cols = 'inc."' + '",\n\t\tinc."'.join(income_df_final.columns) + '"'
bs_cols = 'bs."' + '",\n\t\tbs."'.join(business_df_final.drop(columns='zip').columns) + '"'

SQL_ENGINE_QUERY = f"""
    WITH rate_type_key (rate_type, rate_type_index) AS (
        VALUES
            ('commercial', 0),
            ('industrial', 1),
            ('residential', 2)
    ),
    Utility_Data AS (
        SELECT
            iou.zip, iou.service_type, iou.ownership, rtk.rate_type_index,
            CASE
                WHEN rtk.rate_type = 'commercial' THEN iou.comm_rate
                WHEN rtk.rate_type = 'industrial' THEN iou.ind_rate
                WHEN rtk.rate_type = 'residential' THEN iou.res_rate
            END AS rate
        FROM iou_df iou, rate_type_key rtk
        
        UNION ALL
        
        SELECT
            non_iou.zip, non_iou.service_type, non_iou.ownership, rtk.rate_type_index,
            CASE
                WHEN rtk.rate_type = 'commercial' THEN non_iou.comm_rate
                WHEN rtk.rate_type = 'industrial' THEN non_iou.ind_rate
                WHEN rtk.rate_type = 'residential' THEN non_iou.res_rate
            END AS rate
        FROM non_iou_df non_iou, rate_type_key rtk
    ),
    COMBINED_DATA AS (
        SELECT
            {inc_cols},
            {bs_cols},
            ud.service_type,
            ud.ownership,
            ud.rate_type_index,
            ud.rate
        FROM income_df_final inc
        INNER JOIN business_df_final bs ON inc.zip = bs.zip
        INNER JOIN Utility_Data ud ON ud.zip = inc.zip
        WHERE ud.rate IS NOT NULL AND ud.rate > 0
    )
    SELECT * FROM COMBINED_DATA CD
"""

df_clean = duckdb.sql(SQL_ENGINE_QUERY).df()
print(f"Relational processing successful. Unified matrix layout: {df_clean.shape}")
df_clean.head(5)

## 🛡️ Phase 2: Mitigating Data Challenges, Advanced Feature Engineering, & Data Splitting
With a verified matrix of **181,536 observations**, we deploy systematic feature selection filters and pipeline corrections to satisfy the milestone parameters.

### Resolving the Row-Collapse Bottleneck (Data Challenge):
An intuitive, loop-based Interquartile Range (IQR) filter applied globally across all columns causes an aggressive compounding deletion effect, resulting in an unsustainable **80% data drop-off** (shrinking from 181k down to 36k observations). Because our downstream goal is target tracking (`rate`), we update our pipeline challenge strategy by **exclusively confining our outlier filter to the continuous target vector**.

### Pipeline Tasks Executed Below:
1. **Target Isolation:** Implements targeted outlier trimming on the utility pricing target (`rate`).
2. **Feature Pruning:** Strips away statistical collection noise, dropping the Census Bureau's Margin of Error (`moe_`) attributes and raw IDs.
3. **Categorical Processing:** Transforms structural identifiers (`service_type`, `ownership`) via One-Hot encoding.
4. **Z-Score Normalization:** Scales varied economic distributions into standardized feature spaces via `StandardScaler`.
5. **Deterministic Data Splitting:** Establishes fixed random states to guarantee reproducible **70% Train / 15% Validation / 15% Test** partitions.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# # 1. Targeted Outlier Filtering (Confined strictly to target variable)
target_col = 'rate'
Q1 = df_clean[target_col].quantile(0.25)
Q3 = df_clean[target_col].quantile(0.75)
IQR = Q3 - Q1

lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

df_modeling = df_clean[(df_clean[target_col] >= lower_fence) & (df_clean[target_col] <= upper_fence)].copy()
print("--- Outlier Filtering Summary ---")
print(f"Original Row Count: {df_clean.shape[0]}")
print(f"Filtered Row Count: {df_modeling.shape[0]}")

# # 2. Strategic Feature Selection
feature_cols = [col for col in df_modeling.columns if not col.startswith('moe_') and col not in ['zip', 'rate', 'rate_type_index']]
print(f"\nSelected {len(feature_cols)} true semantic features for predictive modeling.")

# # 3. Categorical Encoding
X_encoded = pd.get_dummies(df_modeling[feature_cols], columns=['service_type', 'ownership'], drop_first=True)
y = df_modeling[target_col]

# # 4. Deterministic Data Partitioning (70/15/15) - PERFORMED FIRST
# Split into train and a temporary validation/test block
X_train_raw, X_temp_raw, y_train, y_temp = train_test_split(X_encoded, y, test_size=0.30, random_state=42)
# Partition the validation and test matrices evenly out of the 30% remainder
X_val_raw, X_test_raw, y_val, y_test = train_test_split(X_temp_raw, y_temp, test_size=0.50, random_state=42)

# # 5. Continuous Feature Scaling - LEAKAGE PREVENTION FIX
continuous_cols = [col for col in X_encoded.columns if not col.startswith(('service_type_', 'ownership_'))]

# Initialize and fit the scaler strictly on the training set parameters
scaler = StandardScaler()
X_train = X_train_raw.copy()
X_train[continuous_cols] = scaler.fit_transform(X_train_raw[continuous_cols])

# Transform validation and test arrays using the training split's mean and variance
X_val = X_val_raw.copy()
X_val[continuous_cols] = scaler.transform(X_val_raw[continuous_cols])

X_test = X_test_raw.copy()
X_test[continuous_cols] = scaler.transform(X_test_raw[continuous_cols])

print(f"\n--- Final Modeling Pipeline Matrix Splits ---")
print(f"X_train Matrix shape : {X_train.shape} | y_train Vector shape : {y_train.shape}")
print(f"X_val Matrix shape   : {X_val.shape}  | y_val Vector shape   : {y_val.shape}")
print(f"X_test Matrix shape  : {X_test.shape}  | y_test Vector shape  : {y_test.shape}")

## 📊 Phase 3: Exploratory Data Analysis & Multi-Panel Visualizations
With our clean, outlier-filtered dataset isolated, we conduct Exploratory Data Analysis to uncover the structural relationships driving utility rates. Per the milestone criteria, we visualize feature distributions, examine socioeconomic correlations, and interpret the patterns surfaced across utility ownership boundaries.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl
import numpy as np       
import pandas as pd      

# Disable LaTeX math parsing bugs globally for this figure
mpl.rcParams['text.usetex'] = False
mpl.rcParams['mathtext.default'] = 'regular'

# Initialize a clean, spacious 2x2 grid layout
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(16, 15))

# --- Plot 1: Target Variable Pricing Density by Ownership Type ---
sns.histplot(
    data=df_modeling, 
    x="rate", 
    hue="ownership", 
    kde=True, 
    element="step", 
    stat="density", 
    common_norm=False, 
    palette="muted",
    ax=axes[0, 0]
)
axes[0, 0].set_title("1. U.S. Electricity Rate Density by Ownership Type", fontsize=12, fontweight="bold")
axes[0, 0].set_xlabel("Rate (Cents per kWh)")
axes[0, 0].set_ylabel("Probability Density")

# --- Plot 2: Socioeconomic Stratification Heatmap ---
income_brackets = [col for col in df_modeling.columns if col.startswith("households_") and "moe_" not in col and "percent_" not in col]
corr_cols = income_brackets + ["rate"]
corr_matrix = df_modeling[corr_cols].corr()
clean_labels = [col.replace('_', ' ') for col in corr_cols]

sns.heatmap(
    corr_matrix, 
    annot=True, 
    cmap="coolwarm", 
    fmt=".2f", 
    linewidths=0.5, 
    xticklabels=clean_labels, 
    yticklabels=clean_labels,
    cbar_kws={'label': 'Pearson Correlation'},
    ax=axes[0, 1]
)
axes[0, 1].set_title("2. Correlation: Income Brackets vs. Rate", fontsize=12, fontweight="bold")
# Tighter rotation and right alignment explicitly prevents text from bleeding into the lower subplots
axes[0, 1].set_xticklabels(axes[0, 1].get_xticklabels(), rotation=35, ha='right', fontsize=9)

# --- Plot 3: Business Density Impact (Commercial Fabric) ---
business_summary_cols = ['est', 'n<5', 'n5_9', 'n10_19', 'n1000']
df_log_biz = df_modeling[business_summary_cols].copy().apply(lambda x: np.log1p(x))
df_log_biz['rate'] = df_modeling['rate']

sns.scatterplot(
    data=df_log_biz,
    x="est",
    y="rate",
    alpha=0.4,
    color="teal",
    ax=axes[1, 0]
)
axes[1, 0].set_title("3. Impact of Total Business Establishment Density (Log Scale)", fontsize=12, fontweight="bold")
axes[1, 0].set_xlabel("Log(Total Establishments + 1)")
axes[1, 0].set_ylabel("Rate (Cents per kWh)")

# --- Plot 4: Target Rate Distribution by Service Type ---
sns.boxplot(
    data=df_modeling,
    x="service_type",
    y="rate",
    hue="service_type",  
    palette="Set2",
    legend=False,        
    ax=axes[1, 1]
)
axes[1, 1].set_title("4. Price Range Distribution Across Service Frameworks", fontsize=12, fontweight="bold")
axes[1, 1].set_xlabel("Service Type Framework")
axes[1, 1].set_ylabel("Rate (Cents per kWh)")

# Explicitly add vertical and horizontal padding between subplots to fix the layout overlap
plt.subplots_adjust(hspace=0.4, wspace=0.35, bottom=0.1)

plt.show()